# Stage 3 CLIP Hybrid Broken-Aware Clean


In [ ]:
import json
import shutil
import subprocess
import traceback
from pathlib import Path
from collections import Counter

RUN_NAME = "stage3_clip_hybrid_broken_aware_clean"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_clip_hybrid_broken_aware_clean.tar.gz")
LOG_PATH = WORK_DIR / "run_log.txt"
ERROR_PATH = WORK_DIR / "error_traceback.txt"

DATASET_ROOT_CANDIDATES = [Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"), Path("/kaggle/input/idid-coco-v3")]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
SEED = 42

def log(msg):
    print(msg, flush=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f: f.write(str(msg) + "\n")

def sh(args, check=True):
    log("$ " + " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], text=True, capture_output=True)
    if p.stdout: log(p.stdout)
    if p.stderr: log(p.stderr)
    if check and p.returncode != 0: raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows

def resolve_crop(row, split_root):
    p=Path(row["crop_path"])
    candidates=[]
    if p.is_absolute(): candidates.append(p)
    candidates += [split_root/p, split_root.parent/p, split_root.parent/"crops"/row.get("split","")/row.get("coarse_class","")/p.name]
    for c in candidates:
        if c.exists(): return c
    raise FileNotFoundError(row.get("crop_path"))

def package_outputs():
    if ARCHIVE_PATH.exists(): ARCHIVE_PATH.unlink()
    sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(WORK_DIR.parent), RUN_NAME], check=False)
    log(f"Archive: {ARCHIVE_PATH}")

try:
    if WORK_DIR.exists(): shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    log(f"RUN_NAME: {RUN_NAME}")
    sh(["nvidia-smi"], check=False)
    DATA_ROOT=None
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_JSONL_REL).exists() and (root/VAL_JSONL_REL).exists(): DATA_ROOT=root; break
    if DATA_ROOT is None: raise FileNotFoundError("stage3_regrouped_v2 not found")
    train_jsonl=DATA_ROOT/TRAIN_JSONL_REL; val_jsonl=DATA_ROOT/VAL_JSONL_REL
    train_root=train_jsonl.parent; val_root=val_jsonl.parent
    log(f"DATA_ROOT: {DATA_ROOT}")
    sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "scikit-learn", "tabulate"])

    import numpy as np, pandas as pd, torch
    from PIL import Image
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import normalize
    from transformers import CLIPModel, CLIPProcessor

    train_rows=[r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
    val_rows=[r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
    log(f"train rows: {len(train_rows)} {Counter(r['coarse_class'] for r in train_rows)}")
    log(f"val rows: {len(val_rows)} {Counter(r['coarse_class'] for r in val_rows)}")
    device="cuda" if torch.cuda.is_available() else "cpu"
    model=CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device); processor=CLIPProcessor.from_pretrained(CLIP_MODEL_ID); model.eval()

    def embed(rows, root, bs=32):
        feats=[]; y=[]; ids=[]
        with torch.no_grad():
            for s in range(0, len(rows), bs):
                b=rows[s:s+bs]
                imgs=[Image.open(resolve_crop(r, root)).convert("RGB") for r in b]
                inp=processor(images=imgs, return_tensors="pt", padding=True).to(device)
                feats.append(model.get_image_features(**inp).detach().cpu().numpy())
                y += [r["coarse_class"] for r in b]; ids += [r["record_id"] for r in b]
        return normalize(np.vstack(feats)), np.array(y), ids

    X_train,y_train,train_ids=embed(train_rows, train_root)
    X_val,y_val,val_ids=embed(val_rows, val_root)

    def metrics(y, pred, prefix=""):
        return {
            prefix+"accuracy": accuracy_score(y,pred),
            prefix+"macro_f1_3class": f1_score(y,pred,labels=LABELS,average="macro",zero_division=0),
            prefix+"ok_recall": ((pred=="insulator_ok")&(y=="insulator_ok")).sum()/max((y=="insulator_ok").sum(),1),
            prefix+"flashover_recall": ((pred=="defect_flashover")&(y=="defect_flashover")).sum()/max((y=="defect_flashover").sum(),1),
            prefix+"broken_recall": ((pred=="defect_broken")&(y=="defect_broken")).sum()/max((y=="defect_broken").sum(),1),
        }

    main_candidates=[]
    for C in [0.03,0.1,0.3,1,3,10]: main_candidates.append((C,"balanced"))
    broken_Cs=[0.03,0.1,0.3,1,3,10]
    broken_thresholds=[0.15,0.20,0.25,0.30,0.35,0.40,0.50,0.60]
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows=[]
    for main_C, main_weight in main_candidates:
        for bC in broken_Cs:
            for thr in broken_thresholds:
                fold=[]
                for tr,dev in cv.split(X_train,y_train):
                    main=LogisticRegression(max_iter=3000,C=main_C,class_weight=main_weight).fit(X_train[tr],y_train[tr])
                    pred=main.predict(X_train[dev])
                    by=(y_train[tr]=="defect_broken").astype(int)
                    broken=LogisticRegression(max_iter=3000,C=bC,class_weight="balanced").fit(X_train[tr],by)
                    bp=broken.predict_proba(X_train[dev])[:,1]
                    hp=pred.copy(); hp[bp>=thr]="defect_broken"
                    fold.append(metrics(y_train[dev],hp))
                row={"main_C":main_C,"main_weight":main_weight,"broken_C":bC,"broken_thr":thr}
                for k in ["accuracy","macro_f1_3class","ok_recall","flashover_recall","broken_recall"]:
                    row["cv_"+k]=float(np.mean([m[k] for m in fold]))
                rows.append(row)
    cv_summary=pd.DataFrame(rows).sort_values(["cv_macro_f1_3class","cv_accuracy"],ascending=False)
    cv_summary.to_csv(WORK_DIR/"train_cv_broken_aware_summary.csv",index=False)
    best=cv_summary.iloc[0].to_dict(); log(best)

    main=LogisticRegression(max_iter=3000,C=float(best["main_C"]),class_weight=best["main_weight"]).fit(X_train,y_train)
    pred=main.predict(X_val)
    broken_y=(y_train=="defect_broken").astype(int)
    broken=LogisticRegression(max_iter=3000,C=float(best["broken_C"]),class_weight="balanced").fit(X_train,broken_y)
    bp=broken.predict_proba(X_val)[:,1]
    hybrid=pred.copy(); hybrid[bp>=float(best["broken_thr"])]="defect_broken"
    val={**best, **metrics(y_val,hybrid,prefix="val_")}
    pd.DataFrame([val]).to_csv(WORK_DIR/"val_summary_broken_aware.csv",index=False)
    pd.DataFrame({"record_id":val_ids,"gt":y_val,"main_pred":pred,"broken_prob":bp,"hybrid_pred":hybrid,"correct":y_val==hybrid}).to_csv(WORK_DIR/"val_predictions_broken_aware.csv",index=False)
    report=classification_report(y_val,hybrid,labels=LABELS,zero_division=0)
    cm=confusion_matrix(y_val,hybrid,labels=LABELS)
    (WORK_DIR/"val_classification_report.txt").write_text(report,encoding="utf-8")
    pd.DataFrame(cm,index=LABELS,columns=LABELS).to_csv(WORK_DIR/"val_confusion_matrix.csv")
    lines=["# Stage 3 CLIP Hybrid Broken-Aware Clean","","Train-CV selects a main 3-class CLIP linear probe plus a one-vs-rest broken rescue threshold.","","## Selected",str(best),"","## Val",pd.DataFrame([val]).to_markdown(index=False),"","```text",report,"```","","## Top train-CV",cv_summary.head(12).to_markdown(index=False)]
    (WORK_DIR/"summary.md").write_text("\n".join(lines),encoding="utf-8")
    log(pd.DataFrame([val]).to_string(index=False))
    package_outputs()
except Exception:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    ERROR_PATH.write_text(traceback.format_exc(),encoding="utf-8")
    print(traceback.format_exc(),flush=True)
    package_outputs()
    raise
